# Build a Deep Research Agent over a Private Knowledge Base — Oracle AI Database + Claude

This notebook builds a **deep research agent**: the flagship use case for the
[`deepagents`](https://pypi.org/project/deepagents/) framework. The agent researches a
**private knowledge base** stored in the **Oracle AI Database** and is driven by **Anthropic's
Claude** through Oracle `langchain-oci`'s *bring-your-own-model* factory.

## The use case, in plain terms

Imagine an **internal research analyst** at a company. You hand them a broad, vague question —
*"How should we ground and operate our next-generation AI agents?"* — and a filing cabinet of
internal documents. A good analyst doesn't blurt out an answer. They:

1. **Break the question into sub-topics** (retrieval, memory, evaluation, risks, recommendation).
2. **Dig through the documents** for evidence on each sub-topic, one at a time.
3. **Keep organised notes** as they go, instead of holding everything in their head.
4. **Double-check** their draft against the sources before writing the final brief.
5. **Deliver a tight, cited report** that a busy executive can act on.

A plain chatbot can't do this reliably: it tries to answer in one shot, forgets constraints over
a long task, and — worst of all — **makes things up** when it can't find an answer. A **deep
agent** is built exactly for this shape of work. In this notebook we build that analyst, give it
the company's documents (in the Oracle AI Database), and watch it produce a grounded, cited brief.

> **Our scenario:** *Aurora Financial* is building an enterprise AI agent platform. Its internal
> memos (engineering, strategy, governance, an incident postmortem) form the knowledge base. The
> agent must answer **only** from those memos and cite each one — because in a regulated industry,
> a confident but ungrounded answer is a liability.

## The four powers of a deep agent

The agent demonstrates the four capabilities that make an agent *deep* rather than a
chat-with-tools loop:

1. **Planning** — it writes and tracks a to-do list before acting.
2. **A virtual filesystem** — it drafts long artifacts to "files" in its own state, off the chat.
3. **Sub-agents** — it delegates focused retrieval to isolated helpers, then a critic verifies.
4. **Streaming** — you can watch the plan → act → observe loop unfold step by step.

## Architecture at a glance

| Component | Role | Where it runs |
|---|---|---|
| Knowledge base / vector store | grounds every answer | 🟢 **Local Docker** — Oracle AI Database Free (gvenzl image) |
| Embeddings (ingest + query) | text → vectors | 🟢 **Local** — HuggingFace `all-MiniLM-L6-v2` (CPU) |
| Retrieval | hybrid (vector + keyword) search | 🟢 **In-database** — Oracle AI Vector Search + Oracle Text |
| Reasoning / planning brain | drives the deep-agent loop | ☁️ **Claude** via `langchain-anthropic` |
| Agent harness | planning, files, sub-agents | `langchain-oci` → `deepagents` |

> **Why Claude for the brain?** A deep agent is "an LLM in a loop." The hard part — decomposing a
> task, delegating, and synthesizing — lives in the model. The `deepagents` harness is modelled on
> Claude Code, so a strong tool-caller like Claude drives it reliably. The **data stays local** in
> the Oracle AI Database; only the reasoning is hosted. Part 5 shows how to swap in a local model.

## Key terms (read once, refer back as needed)

| Term | What it means here |
|---|---|
| **Deep agent** | An LLM given a *planning tool*, a *virtual filesystem*, and the ability to spawn *sub-agents* — scaffolding for long, multi-step tasks. |
| **RAG** (Retrieval-Augmented Generation) | Grounding answers in documents fetched at query time, instead of relying on the model's training memory. |
| **Agentic RAG** | The agent itself decides *when* and *what* to retrieve, can retrieve several times, and *verifies* evidence before answering. |
| **Embedding / vector** | A list of numbers capturing a piece of text's *meaning*. Similar meanings produce nearby vectors. |
| **Vector search** | Finding documents whose embeddings are closest to the query's embedding — i.e. semantic similarity. |
| **Hybrid search** | Combining *vector* search (meaning) with *keyword* search (exact terms), fused via Reciprocal Rank Fusion (RRF). |
| **Grounding** | Requiring every factual claim to be backed by a retrieved document, with a citation. |
| **Sub-agent** | A scoped helper agent with its own prompt and tools; isolates context so the main agent stays focused. |
| **Bring-your-own-model (BYO)** | Handing the agent factory *any* chat model — decoupling the reasoning brain from the data infrastructure. |

> **🔑 Orientation takeaway:** We are building an *agentic-RAG deep agent*. It plans, retrieves
> from the Oracle AI Database (hybrid search), verifies, and writes a cited brief — all driven by a
> Claude brain that we "bring" to the Oracle harness.

# Part 1 · Stand up the Oracle AI Database

Before any agent, we need a database that can store text **and** its vector embeddings, and search
both. We use the **Oracle AI Database Free** edition running locally in Docker via the popular,
community-maintained **gvenzl** image.

## Prerequisites

### 1. Oracle AI Database in Docker (gvenzl image)

```bash
docker run -d --name oracle-free \
  -p 1521:1521 --shm-size=2g \
  -e ORACLE_PASSWORD=Welcome12345 \
  gvenzl/oracle-free
```

- `gvenzl/oracle-free` is the lightweight, community image of the **Oracle AI Database Free**
  edition (maintained by Gerald Venzl). It is much smaller and faster to start than the official
  image.
- Wait for `DATABASE IS READY TO USE` (`docker logs -f oracle-free`).
- Service `FREEPDB1` → DSN `localhost:1521/FREEPDB1`. The `SYSTEM` password is the
  `ORACLE_PASSWORD` you set.
- ⚠️ Use the **full** image (`gvenzl/oracle-free`), *not* `:slim` — the slim variant strips
  **Oracle Text**, which we need for the keyword half of hybrid search.

### 2. Python 3.11–3.13

The `deepagents` package requires Python 3.11–3.13.

### 3. An Anthropic API key

Export `ANTHROPIC_API_KEY=sk-ant-...` (or drop it in a local `.env`; the key cell below also
prompts with `getpass`). Every agent run makes **live calls to the Anthropic API**.

## 1.1 · Install dependencies

`langchain-oci` is installed from this repo's local source (editable) — it carries the
*bring-your-own-model* deepagents factory we rely on. `langchain-oracledb` provides the Oracle
vector-store backend (`OracleVS`). Paths are relative to this notebook (`samples/11-deepagents/`).

**Key terms:** *editable install* (`pip install -e`) points Python at source on disk so local
changes take effect immediately.

In [1]:
# langchain-oci (BYO-model deepagents factory + Oracle datastores) from local source:
%pip install -q -e ../../libs/oci

# Oracle vector-store backend + driver:
%pip install -q langchain-oracledb oracledb langchain-community

# Local embeddings, the Claude chat model, and the REAL deepagents package
# (requires Python 3.11–3.13; sentence-transformers pulls in torch).
%pip install -q langchain-huggingface sentence-transformers langchain-anthropic "deepagents>=0.6" 

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


## 1.2 · Configuration

Everything is parameterised by environment variables with sensible local defaults, so you can
point the notebook at a different database, embedding model, or Claude model without editing code.

**Key takeaway:** the *data plane* (Oracle, embeddings) is local; only the *reasoning model* is a
hosted Claude.

In [2]:
import os

# --- Local Oracle AI Database (Docker) ---
ORACLE_DSN        = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
ORACLE_ADMIN_USER = os.environ.get("ORACLE_ADMIN_USER", "SYSTEM")
ORACLE_ADMIN_PWD  = os.environ.get("ORACLE_ADMIN_PWD", "Welcome12345")  # = ORACLE_PASSWORD
APP_USER          = os.environ.get("ORACLE_APP_USER", "DEEPAGENTS")
APP_PWD           = os.environ.get("ORACLE_APP_PWD", "Welcome12345")
TABLE_NAME        = os.environ.get("ORACLE_TABLE", "RESEARCH_DOCS")

# --- Local embeddings (HuggingFace, runs on CPU) ---
HF_EMBED_MODEL    = os.environ.get("HF_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
HF_EMBED_DEVICE   = os.environ.get("HF_EMBED_DEVICE", "cpu")  # "cuda" / "mps" if available

# --- Reasoning brain (Claude) ---
CLAUDE_MODEL      = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")

print("DB        :", ORACLE_DSN, "| app user:", APP_USER, "| table:", TABLE_NAME)
print("Embeddings:", HF_EMBED_MODEL, "(local,", HF_EMBED_DEVICE + ")")
print("Brain     :", CLAUDE_MODEL, "(Anthropic)")

DB        : localhost:1521/FREEPDB1 | app user: DEEPAGENTS | table: RESEARCH_DOCS
Embeddings: sentence-transformers/all-MiniLM-L6-v2 (local, cpu)
Brain     : claude-sonnet-4-6 (Anthropic)


## 1.3 · Load the Anthropic API key

We look for `ANTHROPIC_API_KEY` in the environment, then in a local `.env`, and finally prompt for
it with `getpass` so the key never lands in notebook output or source control.

**Key takeaway:** keep secrets out of code — env var first, `.env` second, masked prompt last.

In [3]:
import os, getpass
from pathlib import Path

if not os.environ.get("ANTHROPIC_API_KEY"):
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line.startswith("ANTHROPIC_API_KEY=") and not line.startswith("#"):
                os.environ["ANTHROPIC_API_KEY"] = line.split("=", 1)[1].strip().strip('"').strip("'")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
print("Anthropic API key loaded.")

Anthropic API key loaded.


## 1.4 · Verify the database is reachable

AI Vector Search (the `VECTOR` data type, vector indexes, distance metrics) requires a recent
Oracle AI Database. We connect as the admin user and assert the major version is 23 or higher.

**Key takeaway:** a quick version assert fails fast with a clear message instead of a cryptic
`VECTOR`-related SQL error later.

In [4]:
import oracledb

with oracledb.connect(user=ORACLE_ADMIN_USER, password=ORACLE_ADMIN_PWD, dsn=ORACLE_DSN) as con:
    print("Connected to Oracle:", con.version)
    major = int(con.version.split(".")[0])
    assert major >= 23, (
        f"Oracle {con.version} has no AI Vector Search. Use the Oracle AI Database Free "
        "image (gvenzl/oracle-free)."
    )
print("✓ AI Vector Search available.")

Connected to Oracle: 23.26.0.0.0
✓ AI Vector Search available.


## 1.5 · Create a dedicated application user

Best practice is to keep app data out of `SYSTEM`. We create a `DEEPAGENTS` schema with the
developer role, which includes the privileges to create tables, vector indexes, **and** Oracle
Text search indexes. The block is idempotent — re-running skips the user if it already exists.

**Key term:** a *schema* is a user that owns database objects (tables, indexes).

> **🔑 Part 1 takeaway:** You now have a local **Oracle AI Database** running, verified for AI
> Vector Search, with a clean `DEEPAGENTS` schema ready to hold the knowledge base. The reasoning
> model (Claude) is configured but not yet used — Part 1 is pure data infrastructure.

In [5]:
import oracledb

statements = [
    f'CREATE USER {APP_USER} IDENTIFIED BY "{APP_PWD}"',
    f"GRANT DB_DEVELOPER_ROLE TO {APP_USER}",
    f"GRANT CREATE SESSION TO {APP_USER}",
    f"GRANT UNLIMITED TABLESPACE TO {APP_USER}",
]

with oracledb.connect(user=ORACLE_ADMIN_USER, password=ORACLE_ADMIN_PWD, dsn=ORACLE_DSN) as con:
    cur = con.cursor()
    for stmt in statements:
        try:
            cur.execute(stmt)
            print("ok  :", stmt)
        except oracledb.DatabaseError as exc:
            (err,) = exc.args
            if err.code == 1920:  # ORA-01920: user name already exists
                print("skip:", stmt, "(already exists)")
            else:
                raise
    con.commit()
print(f"✓ App user {APP_USER} ready.")

skip: CREATE USER DEEPAGENTS IDENTIFIED BY "Welcome12345" (already exists)
ok  : GRANT DB_DEVELOPER_ROLE TO DEEPAGENTS
ok  : GRANT CREATE SESSION TO DEEPAGENTS
ok  : GRANT UNLIMITED TABLESPACE TO DEEPAGENTS
✓ App user DEEPAGENTS ready.


# Part 2 · Load and index the private knowledge base

With the database ready, Part 2 turns Aurora's documents into something an agent can search by
*meaning* and by *exact term*. Three steps: build an embedding model, load the corpus into the
Oracle AI Database as vectors, and expose retrieval as agent **tools**.

## 2.1 · Build the local embedding model

**Embeddings** turn text into vectors (lists of numbers) so that similar meanings sit close
together in vector space. Retrieval then becomes "find the document vectors nearest the query
vector." We use `all-MiniLM-L6-v2` — small, fast, 384-dimensional, runs on CPU, needs no API.

The **same** model object is used for ingestion and querying, so documents and queries live in the
same vector space and the dimensions match the `VECTOR` column.

**Key takeaway:** use *one* embedding model for both indexing and querying. For stronger
production retrieval, swap in a larger model (e.g. `nomic-ai/nomic-embed-text-v1.5`,
`BAAI/bge-large-en-v1.5`, or an OCI GenAI embedding) — nothing else changes.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=HF_EMBED_MODEL,
    model_kwargs={"device": HF_EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

dim = len(embedding_model.embed_query("hello, oracle vector search"))
print(f"✓ Local embeddings ready — {HF_EMBED_MODEL} on {HF_EMBED_DEVICE}, {dim}-dim vectors.")

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_eco/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9610.12it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Local embeddings ready — sentence-transformers/all-MiniLM-L6-v2 on cpu, 384-dim vectors.


## 2.2 · The private knowledge base

This is Aurora Financial's internal corpus: short engineering, strategy, and governance memos
about how they build their AI agent platform. The deep agent will answer **only** from these
documents and cite them by `id`.

Each record is a plain dict the `ADB` datastore accepts directly: `id`, `title`, `source`,
`content`. In a real deployment these would be your wikis, runbooks, tickets, and design docs.

**Key takeaway:** the corpus *is* the agent's world. Good IDs and titles make citations clean;
clear, self-contained passages make grounding reliable.

In [7]:
SAMPLE_DOCS = [
    {
        "id": "kb-001",
        "title": "Retrieval benchmark: vector vs keyword vs hybrid",
        "source": "ml-benchmarks",
        "content": "Internal benchmark on 1,200 labelled support questions. Pure vector search scored recall@5 of 0.71: strong on paraphrased and conceptual queries, but it missed exact identifiers such as product codes, fee schedule names, and account types. Keyword search (Oracle Text) was the opposite: precise on exact terms, weak on synonyms and intent. Hybrid search that fuses vector and keyword results with Reciprocal Rank Fusion (RRF) scored the best overall recall@5 of 0.86. Recommendation: default Aurora Assist retrieval to hybrid search."
    },
    {
        "id": "kb-002",
        "title": "Why we chose Oracle AI Vector Search",
        "source": "platform-architecture",
        "content": "We co-locate embeddings with operational data in the Oracle AI Database using the native VECTOR data type and an HNSW vector index. This removed a separate, standalone vector database from our stack. Because vectors live next to customer and product tables, a single SQL statement can combine semantic similarity with ordinary predicates and joins, for example filtering retrieved passages by the requesting customer's entitlements. Benefits: less data movement, one security and backup model, and lower infrastructure cost."
    },
    {
        "id": "kb-003",
        "title": "Agent memory architecture: three tiers",
        "source": "platform-architecture",
        "content": "Aurora Assist uses three memory tiers. Short-term memory is the live conversation buffer. Working memory is a virtual filesystem the agent writes to while solving a task (drafts, notes, intermediate findings) so large artifacts stay out of the chat context. Long-term memory stores durable facts and summaries of past tasks as vectors in Oracle, retrieved on demand. A summarization step compresses long histories to keep the context window bounded on multi-step runs."
    },
    {
        "id": "kb-004",
        "title": "Grounding policy: retrieve before you answer",
        "source": "governance-policy",
        "content": "Mandatory policy for all production agents. Any factual claim about a customer, product, fee, or policy must be supported by a retrieved document, and the answer must cite that document's ID. If retrieval returns no relevant evidence, the agent must say it does not know rather than answer from the model's parametric knowledge. This policy exists to prevent confident, ungrounded answers in regulated contexts."
    },
    {
        "id": "kb-005",
        "title": "Postmortem: the May fee-waiver hallucination",
        "source": "incident-postmortem",
        "content": "An agent told a customer about a fee-waiver program that does not exist, triggering a complaint. Root cause: single-shot RAG retrieved no relevant passages, and the model answered from memory anyway. Corrective actions shipped: (1) enforce the retrieve-before-you-answer grounding policy with an explicit admit-when-unsure instruction; (2) switch to hybrid retrieval to raise recall; (3) add a verification (critique) step that checks every claim against the knowledge base before the answer is returned to the customer."
    },
    {
        "id": "kb-006",
        "title": "Evaluation harness and release gating",
        "source": "eng-strategy",
        "content": "We maintain a golden set of 300 question-answer pairs, each labelled with the documents that must be cited. A nightly regression run scores three metrics: groundedness (are claims supported by the cited docs), answer correctness, and retrieval recall. A release is blocked if groundedness regresses against the previous build. Five percent of production transcripts are sampled for human review each week. Evaluation and observability were funded before expanding autonomy."
    },
    {
        "id": "kb-007",
        "title": "Cost and latency of long agent loops",
        "source": "ml-benchmarks",
        "content": "Multi-step planning with sub-agent delegation costs roughly three to five times the tokens of a single-shot answer and adds noticeable latency. Mitigations we adopted: cap the number of research iterations per task; use a smaller, cheaper model for routine sub-agent retrieval while reserving a stronger model for planning and final synthesis; cache embeddings; and stream partial output to the UI so perceived latency stays low even when the full task takes longer."
    },
    {
        "id": "kb-008",
        "title": "Sub-agents and context isolation",
        "source": "platform-architecture",
        "content": "Answer quality on complex, multi-part questions drops when a single context window holds every retrieved chunk for every sub-topic. Aurora delegates each research sub-topic to an isolated sub-agent that does its own retrieval and returns only distilled, cited findings. The orchestrator's context therefore stays focused on planning and synthesis. This pattern measurably improved coherence and reduced dropped constraints on long-horizon tasks."
    },
    {
        "id": "kb-009",
        "title": "Data governance and access control",
        "source": "governance-policy",
        "content": "Embeddings inherit the same row-level security as the source records because they live in Oracle alongside them. Agents run under a service identity scoped to the caller's entitlements, so retrieval cannot surface documents the user is not allowed to see. PII is masked at retrieval time, and every agent tool call is logged for audit. Co-location avoids the risk of syncing access-control lists to a separate external vector store."
    },
    {
        "id": "kb-010",
        "title": "Human-in-the-loop and output guardrails",
        "source": "governance-policy",
        "content": "High-impact actions such as account changes and payments require human approval: the agent proposes an action and a human confirms it through an interrupt before anything executes. Output filters block unsupported financial advice. We deliberately gave autonomy first to bounded, high-volume, low-risk workflows and kept a human in the loop everywhere the blast radius is large."
    },
    {
        "id": "kb-011",
        "title": "Platform roadmap and recommendation",
        "source": "eng-strategy",
        "content": "Recommendation to leadership: standardize on hybrid retrieval over the Oracle AI Database; adopt the three-tier memory model; continue to fund evaluation and observability before widening autonomy; prefer several narrow sub-agents over one monolithic agent; and keep humans in the loop for high-impact actions. The next pilot is agentic RAG for multi-hop questions that a single retrieval cannot answer."
    },
    {
        "id": "kb-012",
        "title": "Agentic RAG vs single-shot RAG",
        "source": "eng-strategy",
        "content": "Single-shot RAG retrieves once and answers. Agentic RAG lets the agent decide when and what to retrieve, issue several retrieval calls, and verify the evidence before answering. It handles multi-hop and ambiguous questions far better, at higher token cost and latency. Aurora's deep-research agent is an agentic-RAG design: it plans, delegates retrieval to sub-agents, and runs a verification step before it finalizes an answer."
    }
]

print(f"{len(SAMPLE_DOCS)} documents across "
      f"{len(set(d['source'] for d in SAMPLE_DOCS))} sources.")

12 documents across 5 sources.


## 2.3 · Ingest into the Oracle AI Database and enable hybrid search

`ADB` is `langchain-oci`'s Oracle datastore. With just `dsn` / `user` / `password` (no
`wallet_location`) it connects to the local container over Easy-Connect. `connect()` builds the
`OracleVS` backend with our embedding model; `bulk_insert` embeds and stores each record. We drop
the table first so re-runs start clean (the vector dimension is tied to the embedding model).

We then create an **Oracle Text search index** on the `TEXT` column. This enables the keyword half
of **hybrid search** — fusing **vector** similarity (great for meaning and paraphrase) with
**keyword** matching (great for exact identifiers) using Reciprocal Rank Fusion (RRF).

**Key takeaways**
- `ADB` + `OracleVS` store the text, metadata, **and** the `VECTOR` embedding in one table.
- The `CREATE SEARCH INDEX` DDL turns on the keyword side of hybrid search — idempotent here.
- Vectors live *next to* your operational data, so retrieval can be combined with ordinary SQL.

In [8]:
import oracledb
from langchain_oci.datastores import ADB

# Fresh start: drop the table if it exists (dimension is tied to the embedding model).
with oracledb.connect(user=APP_USER, password=APP_PWD, dsn=ORACLE_DSN) as con:
    cur = con.cursor()
    try:
        cur.execute(f"DROP TABLE {TABLE_NAME} PURGE")
        print(f"· reset existing table {TABLE_NAME}")
    except oracledb.DatabaseError as exc:
        if exc.args[0].code != 942:  # ORA-00942: table does not exist
            raise
    con.commit()

store = ADB(
    dsn=ORACLE_DSN,
    user=APP_USER,
    password=APP_PWD,
    table_name=TABLE_NAME,
    datastore_description=(
        "Aurora Financial internal AI platform knowledge base: retrieval, agent "
        "memory, evaluation, governance, incidents, and roadmap memos."
    ),
    chunk_on_write=False,  # short docs -> store each as a single passage
)

store.connect(embedding_model)
ingested = store.bulk_insert(SAMPLE_DOCS, embeddings=[[] for _ in SAMPLE_DOCS])
print(f"✓ Ingested {ingested} documents into {APP_USER}.{TABLE_NAME}")

# Enable Oracle Text keyword search so the `search` tool can run HYBRID retrieval.
cur = store.vectorstore.client.cursor()
try:
    cur.execute(f'CREATE SEARCH INDEX {TABLE_NAME}_TXT_IDX ON {TABLE_NAME}("TEXT")')
    print("✓ Oracle Text index created — hybrid (vector + keyword) search enabled.")
except oracledb.DatabaseError as exc:
    if "ORA-00955" in str(exc):  # name already used by an existing object
        print("· Oracle Text index already exists.")
    else:
        raise
finally:
    cur.close()

print("Store stats:", store.stats())

· reset existing table RESEARCH_DOCS


✓ Ingested 12 documents into DEEPAGENTS.RESEARCH_DOCS


✓ Oracle Text index created — hybrid (vector + keyword) search enabled.
Store stats: {'store': 'adb', 'table': 'RESEARCH_DOCS', 'document_count': 12, 'sources': {'governance-policy': 3, 'eng-strategy': 3, 'platform-architecture': 3, 'ml-benchmarks': 2, 'incident-postmortem': 1}}


## 2.4 · Sanity-check Oracle retrieval

Before building the agent, confirm the store returns grounded results straight from Oracle AI
Database. You do **not** need to build tools by hand: when we pass `datastores={"kb": store}` to
`create_deepagents_agent` below, the factory builds the **`search`** (hybrid vector + keyword),
**`get_document`**, and **`stats`** tools from this store automatically — and, because we pass our
local `embedding_model`, they make **no OCI calls**.

**Key term:** a *tool* is a function the LLM can decide to call; the framework runs it and feeds
the result back into the conversation.

> **🔑 Part 2 takeaway:** Aurora's documents are now vectors in Oracle AI Database, searchable by
> **meaning and exact term**. The agent will never see SQL or vectors — only the `search` /
> `get_document` / `stats` tools the `datastores=` one-liner wires up — and everything it answers
> can be traced back to a Doc ID.

In [9]:
# Sanity check: retrieval is grounded in the documents we just ingested into Oracle.
# No manual tool-building needed — create_deepagents_agent(datastores=...) below wires the
# search / get_document / stats tools from this same store for you.
hits = store.search_documents("which retrieval approach gives the best recall and why", top_k=3)
print(f"Retrieved {len(hits)} grounded passage(s) from Oracle AI Database. Top match:\n")
print(hits[0].page_content[:300] if hits else "(no results)")

Retrieved 3 grounded passage(s) from Oracle AI Database. Top match:

Internal benchmark on 1,200 labelled support questions. Pure vector search scored recall@5 of 0.71: strong on paraphrased and conceptual queries, but it missed exact identifiers such as product codes, fee schedule names, and account types. Keyword search (Oracle Text) was the opposite: precise on ex


# Part 3 · Build the deep research agent

Now we assemble the analyst. Part 3 defines the agent's helpers (a reflection tool and two
sub-agents), then wires the whole **deepagents** harness to a Claude brain — grounded in the
Oracle tools from Part 2.

## 3.1 · Define the reflection tool and the sub-agents

Two ingredients make the research loop disciplined:

- **`think_tool`** — a no-op "strategic pause." After each search the agent calls it to state what
  it found, what's still missing, and whether to keep digging. Forcing reflection measurably
  improves multi-step research quality.
- **Sub-agents** — scoped agents with their own prompt that *inherit* the orchestrator's tools (so they can search Oracle without any re-wiring). We define two:
  - **`kb-researcher`**: researches *one* sub-topic against the knowledge base and returns
    distilled, cited findings. The orchestrator delegates each sub-topic to a *fresh* instance, so
    noisy retrieved text stays out of the orchestrator's context (**context isolation**).
  - **`critic`**: re-checks the draft's claims against the knowledge base and flags anything
    unsupported — a built-in guard against hallucination, mirroring Aurora's verification step.

**Key term:** *context isolation* — keeping a sub-task's raw, bulky data inside a sub-agent so the
main agent's limited context window stays focused on planning and synthesis.

**Key takeaway:** a `SubAgent` is just `{name, description, system_prompt}` (`tools` is optional — omit it and the sub-agent inherits the orchestrator's tools, here the Oracle `search` tools). The `description` is how the orchestrator decides *when* to delegate, so write it like a job posting.

In [10]:
from langchain_core.tools import tool


@tool
def think_tool(reflection: str) -> str:
    """Record a strategic reflection: what evidence was found, what gaps remain, and whether to
    keep researching or move on. Call this after each search to stay focused."""
    return f"Reflection recorded: {reflection}"


research_subagent = {
    "name": "kb-researcher",
    "description": (
        "Researches ONE narrow sub-topic against the Aurora knowledge base and returns concise, "
        "cited findings. Delegate a single, focused sub-topic per call."
    ),
    "system_prompt": (
        "You are a focused research specialist for Aurora Financial. Use the `search` tool to find "
        "evidence in the knowledge base, `get_document` to pull a document's full text when a "
        "snippet is not enough, and call `think_tool` after searching to assess what is still "
        "missing. Return a section titled '## Key Findings' with 3-5 concise bullets, each citing a "
        "Doc ID inline like [kb-001]. Use ONLY retrieved content; if the KB has nothing relevant, "
        "say so plainly. No preamble, no conclusion."
    ),
    # tools omitted -> inherits the orchestrator's tools (Oracle search + think_tool)
}

critique_subagent = {
    "name": "critic",
    "description": (
        "Verifies a draft answer's claims against the knowledge base and lists unsupported claims, "
        "missing caveats, or citation errors. Use before finalizing."
    ),
    "system_prompt": (
        "You are a skeptical reviewer. For each claim in the draft you are given, use `search` and "
        "`get_document` to confirm it is supported by the knowledge base. Return a short bullet list "
        "of: unsupported or overstated claims, missing caveats, and citation errors. If everything "
        "checks out, say so briefly. Be concrete and concise."
    ),
    # tools omitted -> inherits the orchestrator's tools (Oracle search)
}

print("Sub-agents defined:", [research_subagent["name"], critique_subagent["name"]])

Sub-agents defined: ['kb-researcher', 'critic']


## 3.2 · Assemble the deep agent — bring-your-own Claude

Two things matter here. First, **bring-your-own-model**: passing `model=` hands the factory **any**
LangChain chat model and bypasses OCI inference auth entirely (the model owns its own connection),
so the **Oracle** harness is driven by **Claude**. Second, **`datastores={"kb": store}`** — the
one-liner that wires Oracle `search` / `get_document` / `stats` into the agent *and* keeps the full
deep-agent harness (planning, files, sub-agents). No manual `create_datastore_tools`.

How context isolation still works:
- The **sub-agents** omit `tools`, so they **inherit** the Oracle search tools and do the retrieval
  in their own isolated contexts.
- The **orchestrator** is told (in its prompt) to plan and delegate rather than search directly, so
  bulky retrieved text stays inside the sub-agents.

The system prompt encodes the classic deep-research workflow: **plan → save the request →
delegate → verify → synthesize**.

**Key takeaways**
- `model=ChatAnthropic(...)` is the entire bring-your-own-model story — `model_id` / OCI auth are ignored.
- Passing `datastores=` or `subagents=` routes through the *real* `deepagents` harness — datastores
  compose with the full agent rather than downgrading it to a lightweight ReAct path.
- `agent._oci_llm` exposes the model actually driving the graph — handy to prove the wiring.

In [11]:
from langchain_anthropic import ChatAnthropic
from langchain_oci import create_deepagents_agent

llm = ChatAnthropic(
    model=CLAUDE_MODEL,
    temperature=0,        # deterministic planning/synthesis
    max_tokens=8000,
    timeout=300,
)

RESEARCH_LEAD_PROMPT = (
    "You are the research lead for Aurora Financial. Produce rigorous, fully-grounded briefs.\n\n"
    "Follow this workflow for every request:\n"
    "1. PLAN: call `write_todos` to break the question into focused sub-topics.\n"
    "2. SAVE: write the user's exact question to `/research_request.md` with your file tools.\n"
    "3. RESEARCH: delegate EACH sub-topic to the `kb-researcher` subagent via the `task` tool — "
    "one sub-topic per call. Do NOT search the knowledge base yourself; the researchers do that.\n"
    "4. VERIFY: assemble a draft, then send it to the `critic` subagent and address its findings.\n"
    "5. SYNTHESIZE: write the final brief to `/final_report.md`, then return it as your answer.\n\n"
    "Rules: every factual claim MUST cite a Doc ID inline like [kb-001]. If the knowledge base does "
    "not support a claim, say so rather than guessing. Keep the final brief tight and executive-ready."
)

agent = create_deepagents_agent(
    model=llm,                                      # <-- bring-your-own Claude (PR #234)
    datastores={"kb": store},                       # one-liner: Oracle retrieval tools + deep harness
    embedding_model=embedding_model,                # local HF embeddings -> no OCI calls
    tools=[think_tool],                             # orchestrator delegates; prompt tells it not to search
    subagents=[research_subagent, critique_subagent],  # sub-agents inherit the Oracle search tools
    system_prompt=RESEARCH_LEAD_PROMPT,
)

print("Agent type:", type(agent).__name__)
print("Driven by the Claude model we passed in:", agent._oci_llm is llm)

Agent type: CompiledStateGraph
Driven by the Claude model we passed in: True


## 3.3 · Display helpers

Small helpers to render what the deep agent produces: the **message flow**, its **plan** (`todos`),
its **virtual files**, and which **sub-agents** it delegated to. Deep-agent state comes back as a
dict with `messages`, `todos`, and `files` keys.

> **🔑 Part 3 takeaway:** The analyst is built. The **deepagents** harness (planning, virtual
> filesystem, sub-agents) is wired to a **Claude** brain via one `model=` argument, and grounded in
> the Oracle tools from Part 2 — with a clean split: the orchestrator plans and delegates, the
> sub-agents retrieve. Its entire working state is inspectable.

In [12]:
from IPython.display import Markdown, display


def show_flow(state):
    print(" -> ".join(type(m).__name__ for m in state["messages"]))


def show_todos(state):
    todos = state.get("todos") or []
    if not todos:
        print("(no plan recorded)"); return
    marks = {"completed": "done", "in_progress": "in&nbsp;progress", "pending": "pending"}
    rows = ["| # | Task | Status |", "|---|------|--------|"]
    for i, t in enumerate(todos, 1):
        rows.append(f"| {i} | {t.get('content','')} | {marks.get(t.get('status'), t.get('status',''))} |")
    display(Markdown("\n".join(rows)))


def show_files(state):
    files = state.get("files") or {}
    if not files:
        print("(no files written)"); return
    for path, blob in files.items():
        content = blob.get("content") if isinstance(blob, dict) else blob
        print(f"FILE {path}  ({len(content)} chars)")
        display(Markdown(content))


def show_delegations(state):
    calls = [tc for m in state["messages"]
             for tc in (getattr(m, "tool_calls", None) or []) if tc["name"] == "task"]
    print(f"{len(calls)} delegation(s) via the built-in `task` tool:\n")
    for tc in calls:
        desc = str(tc["args"].get("description", ""))
        print(f" - [{tc['args'].get('subagent_type')}] {desc[:110]}{'...' if len(desc) > 110 else ''}")

# Part 4 · Run the research and inspect the four powers

Now we give the analyst its assignment and watch it work. We run the task once, then inspect each
of the four deep-agent powers in the single returned state.

## 4.1 · Run the deep research task

A vague executive question that *demands* depth. Watch the message flow — it will be long, because
the orchestrator plans, delegates several sub-topics to `kb-researcher`, runs the `critic`, and
synthesizes. This makes **live Claude API calls** and typically takes a few minutes; that latency
is the deep-agent loop doing real work.

**Key takeaway:** one ambiguous prompt in, a structured + verified + cited brief out — produced by
many small, grounded steps rather than a single guess.

In [13]:
RESEARCH_TASK = (
    "Produce a one-page executive technical brief for Aurora Financial's CTO on how we should "
    "ground and operate our next-generation enterprise AI agents. Cover: (1) our retrieval "
    "strategy, (2) our agent memory approach, (3) evaluation and guardrails, (4) the key "
    "production risks we have learned, and (5) a clear recommendation. Research each angle against "
    "our knowledge base, verify the draft with the critic, and cite document IDs throughout."
)

state = agent.invoke({"messages": [{"role": "user", "content": RESEARCH_TASK}]})
show_flow(state)

HumanMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> ToolMessage -> ToolMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage -> ToolMessage -> AIMessage


## 4.2 · Power #1 — Planning

The plan the agent wrote with `write_todos` and worked through. A deep agent commits to a plan
*before* acting, then updates statuses as it goes — that's what keeps a long task on track.

In [14]:
show_todos(state)

| # | Task | Status |
|---|------|--------|
| 1 | Save the research request to /research_request.md | done |
| 2 | Research sub-topic 1: Retrieval strategy (RAG, chunking, indexing, hybrid search) | done |
| 3 | Research sub-topic 2: Agent memory approach (short-term, long-term, episodic, semantic memory) | done |
| 4 | Research sub-topic 3: Evaluation and guardrails (metrics, safety, monitoring) | done |
| 5 | Research sub-topic 4: Key production risks learned (incidents, failures, lessons) | done |
| 6 | Research sub-topic 5: Recommendations for next-gen enterprise AI agents | done |
| 7 | Assemble draft brief and send to critic subagent for verification | done |
| 8 | Write final verified brief to /final_report.md and return to user | done |

## 4.3 · Power #2 — The virtual filesystem

The documents the agent drafted in its workspace. `/research_request.md` is the saved task;
`/final_report.md` is the brief it composed. These live in **agent state**, not the chat
transcript — which is how deep agents keep long artifacts out of the context window.

In [15]:
show_files(state)

FILE /research_request.md  (581 chars)


# Research Request

**Date:** Current session  
**Audience:** Aurora Financial CTO  
**Document Type:** One-page executive technical brief

## Question
Produce a one-page executive technical brief for Aurora Financial's CTO on how we should ground and operate our next-generation enterprise AI agents. Cover:

1. Our retrieval strategy
2. Our agent memory approach
3. Evaluation and guardrails
4. The key production risks we have learned
5. A clear recommendation

Research each angle against our knowledge base, verify the draft with the critic, and cite document IDs throughout.


FILE /final_report.md  (6778 chars)


# Grounding & Operating Next-Generation Enterprise AI Agents at Aurora Financial
**Executive Technical Brief — CTO**
*All claims grounded in Aurora Financial internal knowledge base. Doc IDs cited inline.*

---

## 1. Retrieval Strategy

Aurora's benchmark across 1,200 labelled support questions [kb-001] established that pure vector search achieves recall@5 = 0.71 — adequate for conceptual queries but brittle on exact identifiers (product codes, fee schedule names, account types). Keyword-only search has the inverse weakness. **Hybrid search** (dense vector + Oracle Text keyword, fused via Reciprocal Rank Fusion) achieves recall@5 = 0.86 and is the recommended default for Aurora Assist [kb-001, kb-011].

Architecturally, Aurora eliminated the standalone vector database in favor of Oracle AI Vector Search with HNSW indexing, co-located with operational data [kb-002]. This single-store design means embeddings inherit row-level security from source records, reducing the risk of ACL desynchronization that would arise from a separate external vector store [kb-009]. A single SQL statement can combine semantic similarity with relational predicates (e.g., customer entitlement filters) in one query.

For complex, multi-hop questions, Aurora is moving from single-shot RAG to **agentic RAG** [kb-012]: the agent plans, issues multiple targeted retrieval calls, and runs a verification step before responding. A post-incident corrective action from the May fee-waiver hallucination [kb-005] added a verification step to the pipeline; the broader agentic RAG architecture is the designated next pilot per the platform roadmap [kb-011].

---

## 2. Agent Memory Approach

Aurora Assist operates a formally defined **three-tier memory model** [kb-003]:

| Tier | Mechanism | Purpose |
|---|---|---|
| Short-term | Live conversation buffer (in-context) | Active dialogue |
| Working | Virtual filesystem (agent-writable) | Artifacts, drafts, intermediate findings — kept out of context window |
| Long-term | Vectors in Oracle AI Database, retrieved on demand | Durable facts, past-task summaries |

A **summarization step** compresses long conversation histories to keep the context window bounded during multi-step runs [kb-003]. On complex multi-part tasks, each sub-topic is delegated to an **isolated sub-agent** that returns only distilled, cited findings to the orchestrator — measurably improving coherence and reducing dropped constraints on long-horizon tasks [kb-008]. The orchestrator's context is reserved for planning and synthesis only.

Critically, the model's **parametric memory is prohibited** for factual claims about customers, products, fees, or policies [kb-004]. The three-tier model is recommended as the platform-wide standard in the Aurora roadmap [kb-011].

---

## 3. Evaluation and Guardrails

**Evaluation harness [kb-006]:** A golden set of 300 question-answer pairs (each annotated with required citation documents) runs nightly. Three metrics are tracked: groundedness, answer correctness, and retrieval recall. A release is **blocked** if groundedness regresses against the prior build. Five percent of live production transcripts are sampled for human review weekly. The explicit organizational principle: *"Evaluation and observability were funded before expanding autonomy."*

**Grounding policy [kb-004]:** Every factual claim must cite a retrieved document. If retrieval returns no evidence, the agent must admit it does not know — no fallback to parametric memory.

**Output guardrails [kb-010]:** High-impact actions (account changes, payments) require explicit human approval via an interrupt-before-execute mechanism. Output filters block unsupported financial advice. Autonomy was deliberately granted first to bounded, high-volume, low-risk workflows.

**Verification step [kb-005, kb-012]:** Agentic RAG pipelines include a built-in critique pass that checks every claim against the knowledge base before the answer is returned to the customer.

---

## 4. Key Production Risks Learned

| Risk | Severity | Evidence | Mitigation |
|---|---|---|---|
| Retrieval failure → hallucination | 🔴 Critical | May fee-waiver incident: agent fabricated a non-existent program when retrieval returned nothing [kb-005] | Hybrid retrieval [kb-001]; grounding policy [kb-004]; verification step [kb-012] |
| Agentic loops: 3–5× token cost + noticeable latency | 🟠 High | Documented cost profile [kb-007] | Iteration caps; tiered model sizing; embedding cache; UI streaming |
| Context window degradation on complex queries | 🟠 High | Dropped constraints on long-horizon tasks [kb-008] | Sub-agent isolation; summarization [kb-003] |
| PII exposure & ACL desynchronization | 🟠 High | Governance policy [kb-009] | Oracle row-level security; PII masking at retrieval; audit logging |
| Premature autonomy expansion | 🟡 Medium | Platform roadmap recommendation [kb-011] | Fund eval/observability before widening autonomy; HITL for high-impact actions [kb-010] |
| Monolithic agent architecture | 🟡 Medium | Platform roadmap recommendation [kb-011] | Prefer narrow, specialized sub-agents over one monolithic agent |

---

## 5. Recommendation

Based on Aurora's documented evidence, the following five-point operating model is recommended for next-generation enterprise agents:

1. **Default to hybrid retrieval.** Standardize on hybrid search (RRF) over Oracle AI Vector Search as the baseline. Pure vector search's 0.71 recall is insufficient for a regulated financial environment where exact identifiers matter [kb-001, kb-011].

2. **Adopt the three-tier memory model as the platform standard.** Short-term buffer + working filesystem + long-term vector store, with summarization for context management and sub-agent isolation for complex tasks [kb-003, kb-008].

3. **Gate every release on groundedness.** The nightly 300-question harness with a hard groundedness block is the minimum viable safety net. Fund evaluation infrastructure before expanding agent autonomy [kb-006, kb-011].

4. **Enforce retrieve-before-answer as a hard policy.** No agent in a regulated context should answer from parametric memory. The May incident is the proof case [kb-004, kb-005].

5. **Expand autonomy incrementally, with HITL for high-impact actions.** Start with bounded, low-risk workflows. Add human-approval interrupts for any action with large blast radius. Pilot agentic RAG for multi-hop questions as the next capability frontier [kb-010, kb-011, kb-012].

---
*Critic-verified. Seven draft issues corrected: two unsupported causal claims removed, two overstated "mandate" framings softened to "recommended," token/latency conflation corrected, "automatically blocked" qualifier removed, and "directive" relabeled as "recommendation" for kb-011.*


## 4.4 · Power #3 — Sub-agents (delegation & context isolation)

Each delegation below ran in its *own* context with its *own* tools and returned only a distilled
result. The noisy retrieved passages stayed inside the researchers; the orchestrator only ever saw
clean findings — then the `critic` verified them.

In [16]:
show_delegations(state)

6 delegation(s) via the built-in `task` tool:

 - [kb-researcher] Research Aurora Financial's internal knowledge base on the topic of RETRIEVAL STRATEGY for enterprise AI agent...
 - [kb-researcher] Research Aurora Financial's internal knowledge base on the topic of AGENT MEMORY APPROACH for enterprise AI ag...
 - [kb-researcher] Research Aurora Financial's internal knowledge base on the topic of EVALUATION AND GUARDRAILS for enterprise A...
 - [kb-researcher] Research Aurora Financial's internal knowledge base on the topic of PRODUCTION RISKS AND INCIDENTS for enterpr...
 - [kb-researcher] Research Aurora Financial's internal knowledge base specifically for STRATEGIC RECOMMENDATIONS and ROADMAP ite...
 - [critic] Please verify the following draft executive brief against the Aurora Financial knowledge base. List any unsupp...


## 4.5 · The final executive brief

The orchestrator's synthesized, critic-reviewed answer — grounded in Aurora's documents and cited
with Doc IDs throughout.

> **🔑 Part 4 takeaway:** From one vague question, the agent produced a **planned, delegated,
> verified, and fully-cited** brief — every claim traceable to a document in the Oracle AI
> Database. That is the payoff of agentic RAG over a single-shot chatbot answer.

In [17]:
display(Markdown(state["messages"][-1].content))

---

# Grounding & Operating Next-Generation Enterprise AI Agents at Aurora Financial
**Executive Technical Brief — CTO**
*Critic-verified. All claims grounded in Aurora Financial internal knowledge base with Doc IDs cited inline.*

---

## 1. Retrieval Strategy

Aurora's benchmark across 1,200 labelled support questions [kb-001] established that pure vector search achieves recall@5 = **0.71** — adequate for conceptual queries but brittle on exact identifiers (product codes, fee schedule names, account types). Keyword-only search has the inverse weakness. **Hybrid search** (dense vector + Oracle Text keyword, fused via Reciprocal Rank Fusion) achieves recall@5 = **0.86** and is the recommended default for Aurora Assist [kb-001, kb-011].

Architecturally, Aurora eliminated the standalone vector database in favor of **Oracle AI Vector Search with HNSW indexing, co-located with operational data** [kb-002]. This single-store design means embeddings inherit row-level security from source records, reducing the risk of ACL desynchronization that would arise from a separate external vector store [kb-009]. A single SQL statement can combine semantic similarity with relational predicates (e.g., customer entitlement filters) in one query.

For complex, multi-hop questions, Aurora is moving from single-shot RAG to **agentic RAG** [kb-012]: the agent plans, issues multiple targeted retrieval calls, and runs a verification step before responding. A post-incident corrective action from the May fee-waiver hallucination [kb-005] added a verification step to the pipeline; the broader agentic RAG architecture is the designated next pilot per the platform roadmap [kb-011].

---

## 2. Agent Memory Approach

Aurora Assist operates a formally defined **three-tier memory model** [kb-003]:

| Tier | Mechanism | Purpose |
|---|---|---|
| **Short-term** | Live conversation buffer (in-context) | Active dialogue |
| **Working** | Virtual filesystem (agent-writable) | Artifacts, drafts, intermediate findings — kept out of context window |
| **Long-term** | Vectors in Oracle AI Database, retrieved on demand | Durable facts, past-task summaries |

A **summarization step** compresses long conversation histories to keep the context window bounded during multi-step runs [kb-003]. On complex multi-part tasks, each sub-topic is delegated to an **isolated sub-agent** that returns only distilled, cited findings to the orchestrator — measurably improving coherence and reducing dropped constraints on long-horizon tasks [kb-008]. The orchestrator's context is reserved for planning and synthesis only.

Critically, the model's **parametric memory is prohibited** for factual claims about customers, products, fees, or policies [kb-004]. The three-tier model is recommended as the platform-wide standard in the Aurora roadmap [kb-011].

---

## 3. Evaluation and Guardrails

**Evaluation harness [kb-006]:** A golden set of **300 question-answer pairs** (each annotated with required citation documents) runs nightly. Three metrics are tracked: groundedness, answer correctness, and retrieval recall. A release is **blocked** if groundedness regresses against the prior build. Five percent of live production transcripts are sampled for human review weekly. The explicit organizational principle: *"Evaluation and observability were funded before expanding autonomy."*

**Grounding policy [kb-004]:** Every factual claim must cite a retrieved document. If retrieval returns no evidence, the agent must admit it does not know — no fallback to parametric memory.

**Output guardrails [kb-010]:** High-impact actions (account changes, payments) require explicit human approval via an **interrupt-before-execute** mechanism. Output filters block unsupported financial advice. Autonomy was deliberately granted first to bounded, high-volume, low-risk workflows.

**Verification step [kb-005, kb-012]:** Agentic RAG pipelines include a built-in critique pass that checks every claim against the knowledge base before the answer is returned to the customer.

---

## 4. Key Production Risks Learned

| Risk | Severity | Evidence | Mitigation |
|---|---|---|---|
| Retrieval failure → hallucination | 🔴 **Critical** | May fee-waiver incident: agent fabricated a non-existent program when retrieval returned nothing [kb-005] | Hybrid retrieval [kb-001]; grounding policy [kb-004]; verification step [kb-012] |
| Agentic loops: 3–5× token cost + noticeable latency | 🟠 **High** | Documented cost profile [kb-007] | Iteration caps; tiered model sizing; embedding cache; UI streaming |
| Context window degradation on complex queries | 🟠 **High** | Dropped constraints on long-horizon tasks [kb-008] | Sub-agent isolation; summarization [kb-003] |
| PII exposure & ACL desynchronization | 🟠 **High** | Governance policy [kb-009] | Oracle row-level security; PII masking at retrieval; audit logging |
| Premature autonomy expansion | 🟡 **Medium** | Platform roadmap recommendation [kb-011] | Fund eval/observability before widening autonomy; HITL for high-impact actions [kb-010] |
| Monolithic agent architecture | 🟡 **Medium** | Platform roadmap recommendation [kb-011] | Prefer narrow, specialized sub-agents over one monolithic agent |

---

## 5. Recommendation

Five-point operating model for next-generation enterprise agents, grounded in Aurora's own evidence:

1. **Default to hybrid retrieval.** Standardize on hybrid search (RRF) over Oracle AI Vector Search. Pure vector search's 0.71 recall is insufficient for a regulated financial environment where exact identifiers matter [kb-001, kb-011].

2. **Adopt the three-tier memory model as the platform standard.** Short-term buffer + working filesystem + long-term vector store, with summarization for context management and sub-agent isolation for complex tasks [kb-003, kb-008].

3. **Gate every release on groundedness.** The nightly 300-question harness with a hard groundedness block is the minimum viable safety net. Fund evaluation infrastructure *before* expanding agent autonomy [kb-006, kb-011].

4. **Enforce retrieve-before-answer as a hard policy.** No agent in a regulated context should answer from parametric memory. The May incident is the proof case [kb-004, kb-005].

5. **Expand autonomy incrementally, with HITL for high-impact actions.** Start with bounded, low-risk workflows. Add human-approval interrupts for any action with large blast radius. Pilot agentic RAG for multi-hop questions as the next capability frontier [kb-010, kb-011, kb-012].

---

> **Process note:** This brief was produced by four parallel KB research agents (one per sub-topic), assembled into a draft, and then verified by a critic agent that identified and corrected seven issues — including two unsupported causal claims, two overstated "mandate" framings (softened to "recommended"), a token/latency conflation, an unverified "automatically" qualifier, and a mislabeling of kb-011 as a "directive" rather than a "recommendation to leadership." The final report is saved to `/final_report.md`.

# Part 5 · Stream the loop and go further

The final power — observability — plus how to extend the pattern to production.

## 5.1 · Power #4 — Stream the plan → act → observe loop

`invoke` waits for the final state. `stream(..., stream_mode="updates")` yields one update per
graph step, so you can watch the agent think, call tools, and observe results in real time — what
you'd surface in a UI.

For a crisp demo we use a **leaner** deep agent built from the *same* factory but with **no
sub-agents** — so it plans and retrieves directly (the single-context variant) and a short question
streams quickly.

**Key takeaway:** the same factory scales down to a simple planning agent and up to a multi-
sub-agent orchestrator — and every step is observable as it happens.

In [18]:
quick_agent = create_deepagents_agent(
    model=llm,
    datastores={"kb": store},            # one line -> Oracle search/get_document/stats + deep harness
    embedding_model=embedding_model,
    tools=[think_tool],                  # no subagents -> it searches directly
    system_prompt=(
        "You are a research assistant for Aurora Financial. Search the knowledge base, then answer "
        "concisely and cite Doc IDs like [kb-001]. If the KB lacks an answer, say so."
    ),
)

print("Watching the agent work, step by step:\n")
for chunk in quick_agent.stream(
    {"messages": [{"role": "user",
                   "content": "In two sentences, what is agentic RAG and why does Aurora use it?"}]},
    stream_mode="updates",
):
    for node, update in chunk.items():
        msgs = update.get("messages") if isinstance(update, dict) else None
        last = msgs[-1] if msgs else None
        kind = type(last).__name__ if last is not None else "-"
        note = ""
        if last is not None and getattr(last, "tool_calls", None):
            note = "  (calls: " + ", ".join(tc["name"] for tc in last.tool_calls) + ")"
        print(f"[{node}] {kind}{note}")

Watching the agent work, step by step:

[PatchToolCallsMiddleware.before_agent] -


[model] AIMessage  (calls: search)
[TodoListMiddleware.after_model] -
[tools] ToolMessage


[model] AIMessage
[TodoListMiddleware.after_model] -


## 5.2 · Cleanup

Close the datastore connection. To tear down the database entirely:

```bash
docker stop oracle-free && docker rm oracle-free
```

In [19]:
store.close()
print("✓ Datastore connection closed.")

✓ Datastore connection closed.


## 5.3 · Recap & key takeaways

- **Deep research over a private store is the canonical deep-agent use case.** We exercised all
  four powers — **planning**, a **virtual filesystem**, **sub-agents**, and **streaming** — on a
  real, ambiguous question and got a structured, verified, cited brief.
- **Grounding = a tool + a strict prompt.** Oracle AI Vector Search (via `ADB` +
  `create_datastore_tools`) is the agent's only source of truth; "retrieve before you answer,
  cite Doc IDs, admit gaps" keeps it honest, and the `critic` sub-agent adds a verification pass.
- **Hybrid retrieval in the database.** Co-locating the `VECTOR` embedding with the text and an
  **Oracle Text** index lets one tool fuse semantic + keyword search (RRF) — strong on both meaning
  and exact identifiers — without bolting on a separate vector database.
- **Bring-your-own-model decouples the brain from the data.** `create_deepagents_agent(model=...)`
  ran the Oracle harness on **Claude** with zero OCI credentials. The data stayed local in the
  Oracle AI Database.

### Variations to try
- **Go fully local:** swap `ChatAnthropic` for a local Ollama model
  (`pip install langchain-ollama`, then `ChatOllama(model="qwen3", temperature=0)`) for a
  zero-cloud stack. Use a strong tool-calling model — deep-agent planning is demanding.
- **Other providers:** `model=ChatOpenAI(model="gpt-5")`, a self-hosted vLLM endpoint, or a
  custom-configured `ChatOCIGenAI`. The agent code is identical.
- **Scale the corpus:** point `ADB` at a real table of wikis/runbooks/tickets and set
  `chunk_on_write=True` for long documents.
- **Persist threads:** pass an Oracle-backed LangGraph checkpointer to keep research sessions
  resumable — mixing the best model with Oracle data infrastructure.

> **🔑 Final takeaway:** A deep agent is a small set of powers — *plan, write to files, delegate to
> sub-agents, stream* — wrapped around an LLM. Point those powers at the **Oracle AI Database** for
> grounding and a **Claude** brain for reasoning, and you have a production-shaped research analyst
> that you can trust because every claim is cited and verified.